# Predicting Student Health Risk
**Kaggle Competition — Multi-class Classification**

Target: `health_condition` → `healthy` | `at-risk` | `unhealthy`

## 1. Environment Setup

In [ ]:
# Install dependencies if running on Kaggle kernels
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

try:
    import optuna
except ImportError:
    pip_install('optuna')

try:
    import lightgbm
except ImportError:
    pip_install('lightgbm')

try:
    import catboost
except ImportError:
    pip_install('catboost')

print('All packages ready.')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    log_loss, roc_auc_score
)
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    VotingClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 110

print('Libraries loaded successfully.')

## 2. Load Data

In [ ]:
import os

# Works both locally and on Kaggle
DATA_DIR = '../data' if os.path.exists('../data') else '/kaggle/input/playground-series-s5e6'

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')
sub   = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

print(f'Train shape : {train.shape}')
print(f'Test shape  : {test.shape}')
train.head()

## 3. Exploratory Data Analysis

In [ ]:
train.info()

In [ ]:
train.describe().T

In [ ]:
# Missing values
def missing_summary(df, name='DataFrame'):
    miss = df.isnull().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    pct  = (miss / len(df) * 100).round(2)
    return pd.DataFrame({'Missing': miss, 'Pct %': pct}).rename_axis(name)

print('=== TRAIN ===')
display(missing_summary(train, 'Train'))
print('\n=== TEST ===')
display(missing_summary(test, 'Test'))

In [ ]:
# Target distribution
fig, ax = plt.subplots(figsize=(7, 4))
vc = train['health_condition'].value_counts()
ax.bar(vc.index, vc.values, color=sns.color_palette('Set2', len(vc)))
for i, (k, v) in enumerate(vc.items()):
    ax.text(i, v + 50, f'{v:,}\n({v/len(train)*100:.1f}%)', ha='center', fontsize=10)
ax.set_title('Target Class Distribution', fontsize=13)
ax.set_xlabel('health_condition')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Numeric features by target
num_cols = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
            'step_count', 'exercise_duration', 'water_intake']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    for label in train['health_condition'].unique():
        subset = train.loc[train['health_condition'] == label, col].dropna()
        axes[i].hist(subset, bins=40, alpha=0.5, label=label, density=True)
    axes[i].set_title(col, fontsize=11)
    axes[i].legend(fontsize=8)
axes[-1].set_visible(False)
fig.suptitle('Numeric Features by Health Condition', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features vs target
cat_cols = ['diet_type', 'stress_level', 'sleep_quality',
            'physical_activity_level', 'smoking_alcohol', 'gender']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    ct = pd.crosstab(train[col], train['health_condition'], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[i], colormap='Set2', edgecolor='white', width=0.7)
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('% within category')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].legend(fontsize=8)
fig.suptitle('Categorical Features vs Health Condition (%)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numeric)
corr = train[num_cols].corr()
fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Numeric Feature Correlations', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Preprocessing & Feature Engineering

In [ ]:
TARGET = 'health_condition'

# Encode target
le = LabelEncoder()
y = le.fit_transform(train[TARGET])
print('Classes:', le.classes_)

# Combine for joint engineering
train['_split'] = 'train'
test['_split']  = 'test'
test[TARGET]    = np.nan
all_data = pd.concat([train, test], axis=0, ignore_index=True)

In [ ]:
def engineer_features(df):
    df = df.copy()

    # Ordinal mappings
    stress_map   = {'low': 0, 'medium': 1, 'high': 2}
    quality_map  = {'poor': 0, 'average': 1, 'good': 2}
    activity_map = {'sedentary': 0, 'moderate': 1, 'active': 2}
    smoke_map    = {'no': 0, 'occasional': 1, 'yes': 2}

    df['stress_ord']   = df['stress_level'].map(stress_map)
    df['quality_ord']  = df['sleep_quality'].map(quality_map)
    df['activity_ord'] = df['physical_activity_level'].map(activity_map)
    df['smoke_ord']    = df['smoking_alcohol'].map(smoke_map)

    # One-hot encode low-cardinality nominals
    df = pd.get_dummies(df, columns=['diet_type', 'gender'], drop_first=False)

    # Interaction / ratio features
    df['active_index']      = df['step_count'] / (df['exercise_duration'] + 1)
    df['calorie_per_step']  = df['calorie_expenditure'] / (df['step_count'] + 1)
    df['sleep_stress']      = df['sleep_duration'] * (3 - df['stress_ord'].fillna(1))
    df['bmi_activity']      = df['bmi'] / (df['activity_ord'].fillna(1) + 1)
    df['hydration_index']   = df['water_intake'] / (df['bmi'] + 1)
    df['health_score']      = (
        df['step_count'] / 10000
        + df['sleep_duration'] / 8
        + df['water_intake'] / 3
        - df['bmi'] / 30
        - df['stress_ord'].fillna(1) / 2
    )

    # Drop original ordinal string columns (already encoded)
    df.drop(columns=['stress_level', 'sleep_quality',
                     'physical_activity_level', 'smoking_alcohol'], inplace=True)
    return df

all_data = engineer_features(all_data)
print('Shape after feature engineering:', all_data.shape)
all_data.head(2)

In [ ]:
# Split back
feature_cols = [c for c in all_data.columns
                if c not in ['id', TARGET, '_split']]

X_train = all_data.loc[all_data['_split'] == 'train', feature_cols].reset_index(drop=True)
X_test  = all_data.loc[all_data['_split'] == 'test',  feature_cols].reset_index(drop=True)

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'Missing in X_train: {X_train.isnull().sum().sum()}')
print(f'Missing in X_test : {X_test.isnull().sum().sum()}')

In [ ]:
# Impute remaining NaNs with KNN
imputer = KNNImputer(n_neighbors=5)
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train),  columns=feature_cols)
X_test_imp  = pd.DataFrame(imputer.transform(X_test),        columns=feature_cols)

print('Imputation done. Remaining NaN in train:', X_train_imp.isnull().sum().sum())

## 5. Baseline Models — Cross-Validation

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

baseline_models = {
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    'XGBoost':      xgb.XGBClassifier(
                        n_estimators=500, learning_rate=0.05,
                        max_depth=6, subsample=0.8,
                        colsample_bytree=0.8, use_label_encoder=False,
                        eval_metric='mlogloss', random_state=SEED,
                        tree_method='hist', device='cpu'
                    ),
    'LightGBM':     lgb.LGBMClassifier(
                        n_estimators=500, learning_rate=0.05,
                        num_leaves=63, subsample=0.8,
                        colsample_bytree=0.8, random_state=SEED,
                        n_jobs=-1, verbose=-1
                    ),
    'CatBoost':     CatBoostClassifier(
                        iterations=500, learning_rate=0.05,
                        depth=6, random_seed=SEED,
                        verbose=0, thread_count=-1
                    ),
}

cv_results = {}
for name, model in baseline_models.items():
    scores = cross_val_score(model, X_train_imp, y, cv=CV,
                             scoring='accuracy', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:15s}  ACC: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# Plot CV results
fig, ax = plt.subplots(figsize=(8, 4))
names  = list(cv_results.keys())
means  = [cv_results[n].mean() for n in names]
stds   = [cv_results[n].std()  for n in names]
colors = sns.color_palette('Set2', len(names))
bars = ax.barh(names, means, xerr=stds, color=colors,
               edgecolor='white', capsize=5)
for bar, m in zip(bars, means):
    ax.text(m + 0.001, bar.get_y() + bar.get_height()/2,
            f'{m:.4f}', va='center', fontsize=10)
ax.set_xlabel('CV Accuracy (5-fold)')
ax.set_title('Baseline Model Comparison')
ax.set_xlim(min(means) - 0.02, max(means) + 0.02)
plt.tight_layout()
plt.show()

## 6. Hyperparameter Tuning with Optuna

In [ ]:
def objective_lgbm(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 300, 1500),
        'learning_rate':   trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'num_leaves':      trial.suggest_int('num_leaves', 31, 255),
        'max_depth':       trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample':       trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':       trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':      trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'random_state':    SEED,
        'n_jobs':          -1,
        'verbose':         -1,
    }
    model = lgb.LGBMClassifier(**params)
    scores = cross_val_score(model, X_train_imp, y, cv=CV,
                             scoring='accuracy', n_jobs=1)
    return scores.mean()

study_lgbm = optuna.create_study(direction='maximize',
                                  sampler=optuna.samplers.TPESampler(seed=SEED))
study_lgbm.optimize(objective_lgbm, n_trials=50, show_progress_bar=True)

print(f'\nBest LightGBM CV accuracy: {study_lgbm.best_value:.4f}')
print('Best params:', study_lgbm.best_params)

In [ ]:
def objective_xgb(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 300, 1500),
        'learning_rate':   trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'max_depth':       trial.suggest_int('max_depth', 3, 10),
        'subsample':       trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'gamma':           trial.suggest_float('gamma', 0, 5),
        'reg_alpha':       trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':      trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'eval_metric':     'mlogloss',
        'use_label_encoder': False,
        'random_state':    SEED,
        'tree_method':     'hist',
        'device':          'cpu',
    }
    model = xgb.XGBClassifier(**params)
    scores = cross_val_score(model, X_train_imp, y, cv=CV,
                             scoring='accuracy', n_jobs=1)
    return scores.mean()

study_xgb = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
study_xgb.optimize(objective_xgb, n_trials=50, show_progress_bar=True)

print(f'\nBest XGBoost CV accuracy: {study_xgb.best_value:.4f}')
print('Best params:', study_xgb.best_params)

## 7. Train Final Models & OOF Predictions

In [ ]:
n_classes = len(le.classes_)

best_lgbm = lgb.LGBMClassifier(**{**study_lgbm.best_params,
                                   'random_state': SEED, 'n_jobs': -1, 'verbose': -1})
best_xgb  = xgb.XGBClassifier(**{**study_xgb.best_params,
                                  'eval_metric': 'mlogloss',
                                  'use_label_encoder': False,
                                  'random_state': SEED,
                                  'tree_method': 'hist', 'device': 'cpu'})

models = {'LightGBM': best_lgbm, 'XGBoost': best_xgb}

oof_preds  = {name: np.zeros((len(X_train_imp), n_classes)) for name in models}
test_preds = {name: np.zeros((len(X_test_imp),  n_classes)) for name in models}

for name, model in models.items():
    print(f'\n--- {name} ---')
    fold_scores = []
    for fold, (tr_idx, val_idx) in enumerate(CV.split(X_train_imp, y)):
        X_tr, X_val = X_train_imp.iloc[tr_idx], X_train_imp.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        model.fit(X_tr, y_tr)

        val_prob = model.predict_proba(X_val)
        oof_preds[name][val_idx] = val_prob

        fold_acc = accuracy_score(y_val, val_prob.argmax(axis=1))
        fold_scores.append(fold_acc)
        print(f'  Fold {fold+1}: ACC={fold_acc:.4f}')

        test_preds[name] += model.predict_proba(X_test_imp) / CV.n_splits

    print(f'  OOF ACC: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')

In [ ]:
# Confusion matrix for best OOF model
best_model_name = max(oof_preds,
    key=lambda n: accuracy_score(y, oof_preds[n].argmax(axis=1)))

y_pred_oof = oof_preds[best_model_name].argmax(axis=1)
cm = confusion_matrix(y, y_pred_oof)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'OOF Confusion Matrix — {best_model_name}')
plt.tight_layout()
plt.show()

print(classification_report(y, y_pred_oof, target_names=le.classes_))

## 8. Ensemble

In [ ]:
# Simple average ensemble
ensemble_oof  = np.mean([oof_preds[n]  for n in models], axis=0)
ensemble_test = np.mean([test_preds[n] for n in models], axis=0)

ens_acc = accuracy_score(y, ensemble_oof.argmax(axis=1))
print(f'Ensemble OOF Accuracy: {ens_acc:.4f}')

# Compare individual vs ensemble
for name in models:
    acc = accuracy_score(y, oof_preds[name].argmax(axis=1))
    print(f'  {name}: {acc:.4f}')
print(f'  Ensemble: {ens_acc:.4f}')

## 9. Feature Importance

In [ ]:
# Refit on full training data to extract importance
best_lgbm.fit(X_train_imp, y)
importances = pd.Series(best_lgbm.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(9, 6))
importances.plot(kind='barh', ax=ax, color=sns.color_palette('viridis', 20))
ax.set_title('Top 20 Feature Importances (LightGBM)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 10. Generate Submission

In [ ]:
final_preds = le.inverse_transform(ensemble_test.argmax(axis=1))

submission = pd.DataFrame({
    'id':               test['id'],
    'health_condition': final_preds
})

submission.to_csv('../outputs/submission.csv', index=False)
print(f'Saved submission.csv  ({len(submission):,} rows)')
submission['health_condition'].value_counts()

In [ ]:
submission.head(10)